# Zomato Analytics — Bronze Layer (Raw Ingestion)

**Purpose**: Ingest raw data from source systems into the Bronze (landing) layer with
minimal transformation. Data is stored as-is to preserve full lineage and enable
reprocessing from source.

**Unity Catalog**: All objects are created under a **dedicated catalog** (`zomato_analytics`)
with schema `bronze`. This ensures **zero impact** on any existing catalogs, schemas, or tables.

**Namespace**: `zomato_analytics.bronze.<table_name>`

**Data Sources**:
- `customers` — customer registration & profile data
- `restaurants` — restaurant onboarding & metadata
- `orders` — order transactions
- `deliveries` — delivery tracking events
- `reviews` — customer reviews & ratings
- `delivery_partners` — **Lakebase (Postgres) OLTP, full or incremental CDC load** (Exercise 5)

**SLA**: Runs every 15 minutes (near real-time) | Full refresh: daily at 02:00 UTC

## Exercise 5 — Lakebase CDC into Bronze: how this solution works

**Business problem.** The delivery-partner master (city, vehicle, status, rating, KYC …) lives in an operational
**Lakebase Postgres** database and changes all day: partners join, move city, upgrade vehicles, get suspended, leave.
We want it in the lakehouse **without copying the whole table every run**, **without losing deletes**, and with
**full history** for analytics. The answer is **CDC — Change Data Capture**: record every change at the source and
move only the changes.

```
  ┌────────────────────────────────────────┐
  │ LAKEBASE  (Postgres OLTP database)     │
  │ zomato_ops.delivery_partners           │  <── partner app writes here
  │   │  (a trigger copies every change)   │
  │   └─> zomato_ops.cdc_changelog         │  one row per INSERT / UPDATE / DELETE
  └────────────────────┬───────────────────┘
                       │  ① BRONZE: copy only the NEW changes (full load = whole-table snapshot)  <── THIS NOTEBOOK
                       ▼
  ┌────────────────────────────────────────┐
  │ bronze.lakebase_delivery_partners_cdc  │  every change received = history
  └────────────────────┬───────────────────┘
                       │  ② SILVER: newest change per partner wins → MERGE
                       ▼
  ┌────────────────────────────────────────┐
  │ silver.delivery_partners               │  1 row per partner = how it looks NOW
  └────────────────────┬───────────────────┘
                       │  ③ GOLD: read what changed in Silver → add a new version
                       ▼
  ┌────────────────────────────────────────┐
  │ gold.dim_delivery_partners             │  1 row per partner per version (SCD Type 2)
  └────────────────────────────────────────┘
```

### Bronze algorithm (section 6A)

```
INCREMENTAL load (default)                         FULL load
1. from_txid = highest txid already in Bronze + 1  1. read every row of zomato_ops.delivery_partners
               (our bookmark)                         (one consistent snapshot)
2. upto_txid = oldest transaction still running    2. overwrite Bronze with these rows (op = SNAPSHOT)
               in Lakebase (everything below it    3. bookmark = the snapshot point, so the next
               is finished)                           incremental run continues from there
3. read change-log rows with
       from_txid <= txid < upto_txid
4. append them to Bronze → the bookmark moves forward by itself
```

### Why a *transaction* bookmark and not "last change number"?

```
time ───────────────────────────────────────────────────────────────────────▶
Transaction 100 ──[ change #500 ]────────────────────────────── commit (slow)
Transaction 101 ────────[ change #501 ]── commit
                                                ▲
                                        Bronze runs here
```

- ✘ **"read changes with number > last number read"** — reads #501 and remembers 501. When #500 commits later it is
  **never read** → silent data loss.
- ✔ **"read only FINISHED transactions"** — transaction 100 is still running, so this run stops before it; the next run
  reads #500 and #501. Postgres tells us the boundary with `pg_snapshot_xmin(pg_current_snapshot())`
  (= oldest transaction still running).
- ✔ **The bookmark lives in Bronze itself** (`max(_cdc_txid)`) — it is saved together with the data. If a run fails,
  nothing is saved and the next run reads the same window again: **no gaps, no duplicates**.

**Two ways to run it** — widget / job parameter `lakebase_load_type` (use the same value in Bronze, Silver and Gold):

| Layer | `full` | `incremental` (default) |
|---|---|---|
| Bronze | snapshot of the whole Lakebase table → **overwrite** | only the new changes → **append** |
| Silver | rebuild from all Bronze rows → **overwrite** | only new Bronze rows → **MERGE** |
| Gold | one version per partner → **overwrite** (history restarts) | only changed partners → **SCD2 MERGE** |
| Use it for | first load, recovery, re-baselining | every scheduled run |

The very first `incremental` run also works: the tables are empty, so everything counts as new.
A `full` load rebuilds from the **current** state: partners already deleted in Lakebase are not included (their old
deliveries no longer find a partner) and Gold history restarts at version 1 — that is why `incremental` is the normal mode.

### 🔐 How Bronze connects to Lakebase securely
- **No secret is stored anywhere** — `WorkspaceClient()` uses the identity running this notebook / job.
- The Postgres password is a **1-hour OAuth token** from `w.postgres.generate_database_credential(...)`, created right
  before connecting and never printed or saved.
- The connection is **encrypted** (`sslmode="require"`) and **read-only** (repeatable-read, read-only transaction).
- The reading identity only needs `SELECT` on `zomato_ops.delivery_partners` and `zomato_ops.cdc_changelog`.
- Full checklist (service principals, grants, secret scopes, CI/CD): see the top of `setup/01_generate_data`.

## 0. Install Dependencies (Lakebase client)

In [ ]:
%pip install -q "psycopg[binary]>=3.1" "databricks-sdk>=0.81.0"
dbutils.library.restartPython()

## 1. Configuration & Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, lit, sha2, concat_ws,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType,
    IntegerType, TimestampType, ArrayType, FloatType,
)
from delta.tables import DeltaTable

## 2. Pipeline Parameters

In [ ]:
# Notebook widgets for parameterized runs
dbutils.widgets.text("env", "dev", "Environment (dev/staging/prod)")
dbutils.widgets.text("catalog_name", "zomato_analytics", "Unity Catalog name")
dbutils.widgets.text("source_path", "/Volumes/zomato_analytics/raw/landing", "Source data path")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"], "Load type")

ENV = dbutils.widgets.get("env")
CATALOG = dbutils.widgets.get("catalog_name")
SOURCE_PATH = dbutils.widgets.get("source_path")
LOAD_TYPE = dbutils.widgets.get("load_type")

# Fixed schema name — all Bronze tables live here
SCHEMA = "bronze"

# Fully qualified namespace: catalog.schema
FQN = f"{CATALOG}.{SCHEMA}"

print(f"Environment  : {ENV}")
print(f"Catalog      : {CATALOG}")
print(f"Schema       : {SCHEMA}")
print(f"Namespace    : {FQN}")
print(f"Source Path  : {SOURCE_PATH}")
print(f"Load Type    : {LOAD_TYPE}")

In [ ]:
# --- Lakebase source (Exercise 5) ---
# `load_type` above is for the file sources; `lakebase_load_type` is for the Lakebase source.
dbutils.widgets.dropdown("enable_lakebase_cdc", "true", ["true", "false"], "Enable Lakebase source")
dbutils.widgets.dropdown("lakebase_load_type", "incremental", ["incremental", "full"], "Lakebase load type")
dbutils.widgets.text("lakebase_project_id", "zomato-ops", "Lakebase project id")
dbutils.widgets.text("lakebase_branch_id", "production", "Lakebase branch id")
dbutils.widgets.text("lakebase_database", "databricks_postgres", "Lakebase Postgres database")

ENABLE_LAKEBASE_CDC = dbutils.widgets.get("enable_lakebase_cdc") == "true"
LAKEBASE_LOAD_TYPE = dbutils.widgets.get("lakebase_load_type")
LAKEBASE_PROJECT_ID = dbutils.widgets.get("lakebase_project_id")
LAKEBASE_BRANCH_ID = dbutils.widgets.get("lakebase_branch_id")
LAKEBASE_DATABASE = dbutils.widgets.get("lakebase_database")
BRONZE_PARTNERS_TABLE = f"{FQN}.lakebase_delivery_partners_cdc"

print(f"Lakebase     : {'enabled' if ENABLE_LAKEBASE_CDC else 'disabled'} ({LAKEBASE_LOAD_TYPE} load)")
print(f"  Source     : projects/{LAKEBASE_PROJECT_ID}/branches/{LAKEBASE_BRANCH_ID} → zomato_ops.delivery_partners")
print(f"  Target     : {BRONZE_PARTNERS_TABLE}")

## 3. Create Dedicated Catalog & Schema (Idempotent)

These statements use `IF NOT EXISTS` — they will **never** modify or drop
any existing catalog, schema, or table in your workspace.

In [ ]:
# Create dedicated catalog — isolated from all other catalogs
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"COMMENT ON CATALOG {CATALOG} IS 'Zomato Analytics DE Project — Isolated catalog for medallion architecture'")

# Create bronze schema inside our catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQN}")
spark.sql(f"COMMENT ON SCHEMA {FQN} IS 'Bronze layer — raw ingested data with audit columns'")

# Set current catalog & schema for this notebook
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"✓ Catalog '{CATALOG}' ready")
print(f"✓ Schema '{FQN}' ready")
print(f"✓ Current context: {CATALOG}.{SCHEMA}")

## 4. Schema Definitions
Explicit schemas enforce data contracts at ingestion time.

In [ ]:
SCHEMAS = {
    "customers": StructType([
        StructField("customer_id", StringType(), False),
        StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True),
        StructField("email", StringType(), True),
        StructField("phone", StringType(), True),
        StructField("city", StringType(), True),
        StructField("locality", StringType(), True),
        StructField("pincode", StringType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("signup_date", StringType(), True),
        StructField("subscription_tier", StringType(), True),
        StructField("is_active", BooleanType(), True),
        StructField("preferred_payment", StringType(), True),
        StructField("preferred_cuisine", StringType(), True),
        StructField("total_orders_lifetime", IntegerType(), True),
        StructField("avg_order_value", DoubleType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
    "restaurants": StructType([
        StructField("restaurant_id", StringType(), False),
        StructField("name", StringType(), True),
        StructField("city", StringType(), True),
        StructField("locality", StringType(), True),
        StructField("address", StringType(), True),
        StructField("pincode", StringType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("cuisines", ArrayType(StringType()), True),
        StructField("restaurant_type", StringType(), True),
        StructField("avg_cost_for_two", IntegerType(), True),
        StructField("rating", FloatType(), True),
        StructField("total_reviews", IntegerType(), True),
        StructField("is_delivering_now", BooleanType(), True),
        StructField("has_online_delivery", BooleanType(), True),
        StructField("has_table_booking", BooleanType(), True),
        StructField("is_premium_partner", BooleanType(), True),
        StructField("onboarded_date", StringType(), True),
        StructField("owner_name", StringType(), True),
        StructField("owner_phone", StringType(), True),
        StructField("gst_number", StringType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
    "orders": StructType([
        StructField("order_id", StringType(), False),
        StructField("customer_id", StringType(), True),
        StructField("restaurant_id", StringType(), True),
        StructField("order_placed_at", StringType(), True),
        StructField("order_status", StringType(), True),
        StructField("payment_method", StringType(), True),
        StructField("payment_status", StringType(), True),
        StructField("platform", StringType(), True),
        StructField("subtotal", DoubleType(), True),
        StructField("discount_pct", DoubleType(), True),
        StructField("discount_amount", DoubleType(), True),
        StructField("coupon_code", StringType(), True),
        StructField("delivery_fee", DoubleType(), True),
        StructField("tax_amount", DoubleType(), True),
        StructField("total_amount", DoubleType(), True),
        StructField("num_items", IntegerType(), True),
        StructField("special_instructions", StringType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
    "deliveries": StructType([
        StructField("delivery_id", StringType(), False),
        StructField("order_id", StringType(), True),
        StructField("delivery_partner_id", StringType(), True),
        StructField("delivery_partner_name", StringType(), True),
        StructField("partner_phone", StringType(), True),
        StructField("vehicle_type", StringType(), True),
        StructField("pickup_timestamp", StringType(), True),
        StructField("delivered_timestamp", StringType(), True),
        StructField("delivery_time_mins", IntegerType(), True),
        StructField("delivery_status", StringType(), True),
        StructField("delivery_distance_km", DoubleType(), True),
        StructField("delivery_rating", IntegerType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
    "reviews": StructType([
        StructField("review_id", StringType(), False),
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("restaurant_id", StringType(), True),
        StructField("rating", IntegerType(), True),
        StructField("review_text", StringType(), True),
        StructField("review_timestamp", StringType(), True),
        StructField("sentiment_label", StringType(), True),
        StructField("is_verified_order", BooleanType(), True),
        StructField("upvotes", IntegerType(), True),
        StructField("has_photo", BooleanType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
}

## 5. Bronze Ingestion Functions

In [ ]:
def ingest_to_bronze(
    table_name: str,
    source_format: str = "parquet",
    merge_keys: list = None,
) -> int:
    """
    Ingest a raw dataset into the Bronze Delta table using Unity Catalog.

    All tables are created under: {CATALOG}.{SCHEMA}.{table_name}

    Adds audit columns:
      - _bronze_loaded_at : ingestion timestamp
      - _source_file      : originating file path
      - _row_hash         : SHA-256 hash for dedup / CDC

    Returns the number of records written.
    """
    source_file = f"{SOURCE_PATH}/{table_name}"
    schema = SCHEMAS.get(table_name)

    # Read raw data with enforced schema
    raw_df = (
        spark.read
        .format(source_format)
        .schema(schema)
        .option("mergeSchema", "true")
        .load(source_file)
    )

    # Add audit / lineage columns
    enriched_df = (
        raw_df
        .withColumn("_bronze_loaded_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn(
            "_row_hash",
            sha2(concat_ws("||", *[col(c) for c in raw_df.columns]), 256),
        )
    )

    # Fully qualified table name: catalog.schema.table
    target_table = f"{FQN}.{table_name}"

    if LOAD_TYPE == "full" or not spark.catalog.tableExists(target_table):
        # Full load — overwrite (creates table if not exists)
        (
            enriched_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )
    else:
        # Incremental merge (upsert)
        delta_target = DeltaTable.forName(spark, target_table)
        pk = merge_keys[0] if merge_keys else f"{table_name[:-1]}_id"
        (
            delta_target.alias("tgt")
            .merge(enriched_df.alias("src"), f"tgt.{pk} = src.{pk}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    record_count = enriched_df.count()
    print(f"  ✓ {target_table}: {record_count:,} records ingested ({LOAD_TYPE})")
    return record_count

## 6. Execute Bronze Ingestion Pipeline

In [ ]:
pipeline_tables = [
    {"table_name": "customers",   "merge_keys": ["customer_id"]},
    {"table_name": "restaurants",  "merge_keys": ["restaurant_id"]},
    {"table_name": "orders",       "merge_keys": ["order_id"]},
    {"table_name": "deliveries",   "merge_keys": ["delivery_id"]},
    {"table_name": "reviews",      "merge_keys": ["review_id"]},
]

total_records = 0
pipeline_status = []

for tbl_cfg in pipeline_tables:
    try:
        count = ingest_to_bronze(**tbl_cfg)
        total_records += count
        pipeline_status.append({"table": tbl_cfg["table_name"], "status": "SUCCESS", "records": count})
    except Exception as e:
        pipeline_status.append({"table": tbl_cfg["table_name"], "status": "FAILED", "error": str(e)})
        print(f"  ✗ {tbl_cfg['table_name']}: FAILED — {e}")

## 6A. Lakebase Source — Full or Incremental Load (Exercise 5)

Implements the Bronze algorithm from the top of this notebook:
**6A.1** secure connection → **6A.2** read from Lakebase → **6A.3** write to Bronze → **6A.4** run it.

### 6A.1 Secure connection helper

In [ ]:
import time

import psycopg
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import EndpointType

# WorkspaceClient() automatically uses the identity running this notebook / job.
# No token or password is written anywhere in this notebook.
w = WorkspaceClient()


def find_read_write_endpoint():
    """Return the read-write compute endpoint of our Lakebase branch."""
    branch_path = f"projects/{LAKEBASE_PROJECT_ID}/branches/{LAKEBASE_BRANCH_ID}"
    for endpoint in w.postgres.list_endpoints(parent=branch_path):
        if endpoint.status.endpoint_type == EndpointType.ENDPOINT_TYPE_READ_WRITE:
            return endpoint
    raise RuntimeError(f"No read-write endpoint found on {branch_path}")


def get_lakebase_connection():
    """
    Open a secure connection to Lakebase:
      1. find the database host (the read-write endpoint)
      2. ask Databricks for a short-lived OAuth token (valid 1 hour) and use it as the password
      3. connect with SSL; retry a few times because an idle database may need a moment to wake up
    """
    endpoint = find_read_write_endpoint()
    user_name = w.current_user.me().user_name

    for attempt in range(1, 6):
        try:
            credential = w.postgres.generate_database_credential(endpoint=endpoint.name)
            return psycopg.connect(
                host=endpoint.status.hosts.host,
                port=5432,
                dbname=LAKEBASE_DATABASE,
                user=user_name,
                password=credential.token,  # short-lived OAuth token, never printed or stored
                sslmode="require",          # encrypted connection only
                autocommit=True,
            )
        except psycopg.OperationalError as error:
            print(f"  … Lakebase not ready yet (attempt {attempt}/5): {error}")
            time.sleep(attempt * 5)
    raise RuntimeError("Could not connect to Lakebase after 5 attempts")

### 6A.2 Read from Lakebase — full snapshot or only the new changes

In [ ]:
# Postgres: every transaction with an id BELOW this number is finished (committed or rolled back)
FINISHED_TXID_SQL = "SELECT pg_snapshot_xmin(pg_current_snapshot())::text::bigint"

# INCREMENTAL: change-log rows of finished transactions we have not read yet
NEW_CHANGES_SQL = """
    SELECT change_seq, txid, op, row_data::text, changed_at
    FROM zomato_ops.cdc_changelog
    WHERE table_name = 'zomato_ops.delivery_partners'
      AND txid >= %s    -- not read yet (our bookmark)
      AND txid <  %s    -- transaction is finished
    ORDER BY change_seq
"""

# FULL: every current row, plus the latest change number of each partner
# (Silver uses that number to ignore older changes it may receive later)
FULL_SNAPSHOT_SQL = """
    SELECT coalesce(last_change.change_seq, 0) AS change_seq,
           to_jsonb(p)::text                    AS row_data,
           p.updated_at                         AS changed_at
    FROM zomato_ops.delivery_partners AS p
    LEFT JOIN (
        SELECT pk_value, max(change_seq) AS change_seq
        FROM zomato_ops.cdc_changelog
        WHERE table_name = 'zomato_ops.delivery_partners'
        GROUP BY pk_value
    ) AS last_change ON last_change.pk_value = p.partner_id
"""


def read_from_lakebase(load_type, from_txid=0):
    """
    Read inside ONE consistent, read-only snapshot of the database.
    Returns (finished_txid, rows); every row = (change_seq, txid, op, json_row, changed_at).
    """
    with get_lakebase_connection() as conn:
        conn.isolation_level = psycopg.IsolationLevel.REPEATABLE_READ  # one consistent point in time
        conn.read_only = True                                          # we never write to the source
        with conn.transaction():
            finished_txid = conn.execute(FINISHED_TXID_SQL).fetchone()[0]
            if load_type == "full":
                snapshot = conn.execute(FULL_SNAPSHOT_SQL).fetchall()
                # txid = finished_txid - 1 → the next incremental run starts exactly at finished_txid
                rows = [(seq, finished_txid - 1, "SNAPSHOT", row, changed_at) for seq, row, changed_at in snapshot]
            else:
                rows = conn.execute(NEW_CHANGES_SQL, (from_txid, finished_txid)).fetchall()
    return finished_txid, rows


def get_bronze_bookmark():
    """Highest transaction id already in Bronze (-1 when Bronze is empty)."""
    last_txid = spark.sql(f"SELECT max(_cdc_txid) FROM {BRONZE_PARTNERS_TABLE}").first()[0]
    return -1 if last_txid is None else last_txid

### 6A.3 Write to Bronze

In [ ]:
from pyspark.sql.functions import from_json, to_timestamp

# Shape of one partner row inside the JSON written by the Lakebase trigger
PARTNER_ROW_SCHEMA = """
    partner_id STRING, partner_name STRING, phone STRING, email STRING, city STRING,
    vehicle_type STRING, partner_status STRING, shift_preference STRING, rating DOUBLE,
    joined_date STRING, kyc_verified BOOLEAN, created_at STRING, updated_at STRING
"""


def to_bronze_dataframe(rows):
    """Lakebase rows → Bronze DataFrame: unpack the JSON row and add the audit columns."""
    rows = [(seq, txid, op, row, changed_at.isoformat()) for seq, txid, op, row, changed_at in rows]
    raw_df = spark.createDataFrame(
        rows, "_cdc_change_seq BIGINT, _cdc_txid BIGINT, _cdc_op STRING, _cdc_payload STRING, _cdc_changed_at STRING"
    )
    return (
        raw_df
        .withColumn("row", from_json(col("_cdc_payload"), PARTNER_ROW_SCHEMA))
        .select(
            "row.*",  # partner_id, partner_name, … exactly as in Lakebase
            "_cdc_op", "_cdc_change_seq", "_cdc_txid",
            to_timestamp("_cdc_changed_at").alias("_cdc_changed_at"),
            "_cdc_payload",
        )
        .withColumn("_source_system", lit("lakebase"))
        .withColumn("_source_file", lit(f"lakebase://{LAKEBASE_PROJECT_ID}/{LAKEBASE_BRANCH_ID}/zomato_ops.delivery_partners"))
        .withColumn("_bronze_loaded_at", current_timestamp())
        .withColumn("_row_hash", sha2(col("_cdc_payload"), 256))
    )


def ingest_lakebase_partners():
    """Full load = snapshot + overwrite. Incremental load = new changes + append."""
    if LAKEBASE_LOAD_TYPE == "full":
        finished_txid, rows = read_from_lakebase("full")
        write_mode = "overwrite"
        print(f"  FULL load        : snapshot of {len(rows):,} partners "
              f"(next incremental run starts at transaction {finished_txid:,})")
    else:
        from_txid = get_bronze_bookmark() + 1
        finished_txid, rows = read_from_lakebase("incremental", from_txid)
        write_mode = "append"
        print(f"  INCREMENTAL load : finished transactions from {from_txid:,} up to {finished_txid:,} "
              f"(not included) → {len(rows):,} new changes")

    if rows:
        to_bronze_dataframe(rows).write.format("delta").mode(write_mode).saveAsTable(BRONZE_PARTNERS_TABLE)
    print(f"  ✓ {BRONZE_PARTNERS_TABLE}: {len(rows):,} rows written ({write_mode})")
    return {"table": "lakebase_delivery_partners_cdc", "status": "SUCCESS", "records": len(rows),
            "load_type": LAKEBASE_LOAD_TYPE}

### 6A.4 Run it

In [ ]:
if ENABLE_LAKEBASE_CDC:
    print(f"\n{'='*50}")
    print(f"Lakebase → {BRONZE_PARTNERS_TABLE}")
    print(f"{'='*50}")
    try:
        result = ingest_lakebase_partners()
        total_records += result["records"]
        pipeline_status.append(result)
    except Exception as e:
        pipeline_status.append({"table": "lakebase_delivery_partners_cdc", "status": "FAILED", "error": str(e)})
        print(f"  ✗ lakebase_delivery_partners_cdc: FAILED — {e}")

## 7. Pipeline Summary & Data Quality Checks

In [ ]:
import json
from datetime import datetime

summary = {
    "pipeline": "bronze_ingestion",
    "catalog": CATALOG,
    "schema": SCHEMA,
    "env": ENV,
    "load_type": LOAD_TYPE,
    "run_timestamp": datetime.utcnow().isoformat(),
    "total_records_ingested": total_records,
    "table_results": pipeline_status,
}

print(json.dumps(summary, indent=2))

In [ ]:
# Basic row-count validation
for tbl_cfg in pipeline_tables:
    tbl = tbl_cfg["table_name"]
    fq_table = f"{FQN}.{tbl}"
    count = spark.table(fq_table).count()
    assert count > 0, f"Data quality check FAILED: {fq_table} has 0 records!"
    print(f"  ✓ {fq_table}: {count:,} rows — PASS")

print(f"\n🟢 Bronze ingestion pipeline completed successfully.")
print(f"   All tables created under: {FQN}.*")

In [ ]:
# Lakebase checks (Exercise 5) — fail the task if the load failed or a change was loaded twice
if ENABLE_LAKEBASE_CDC:
    lakebase_status = [s for s in pipeline_status if s["table"] == "lakebase_delivery_partners_cdc"][0]
    assert lakebase_status["status"] == "SUCCESS", f"Lakebase load FAILED: {lakebase_status}"

    check = spark.sql(f"""
        SELECT count(*)                                                      AS total_rows,
               count(DISTINCT partner_id)                                    AS partners,
               count_if(_cdc_op <> 'SNAPSHOT')                               AS change_events,
               count(DISTINCT CASE WHEN _cdc_op <> 'SNAPSHOT' THEN _cdc_change_seq END) AS distinct_change_events
        FROM {BRONZE_PARTNERS_TABLE}
    """).first()
    assert check.total_rows > 0, f"Data quality check FAILED: {BRONZE_PARTNERS_TABLE} is empty!"
    assert check.change_events == check.distinct_change_events, (
        f"Exactly-once check FAILED: {check.change_events - check.distinct_change_events} changes loaded twice"
    )
    print(f"  ✓ {BRONZE_PARTNERS_TABLE}: {check.total_rows:,} rows for {check.partners:,} partners, "
          f"no change loaded twice — PASS")

## 8. Verify Isolation — List Only Our Objects

In [ ]:
print(f"Tables in {FQN}:")
spark.sql(f"SHOW TABLES IN {FQN}").show(truncate=False)